# 04 - Statistical Analysis
**Project:** Social Media Advertising Analytics  
**Goal:** Validate EDA observations with formal hypothesis testing.

---

## Hypotheses tested

| # | Hypothesis | Test |
|---|---|---|
| H1 | Mean ROI differs across channels | One-way ANOVA |
| H2 | Acquisition Cost correlates with ROI | Pearson correlation |
| H3 | Conversion Rate differs across segments | One-way ANOVA |
| H4 | Engagement Score correlates with Conversion Rate | Pearson correlation |

**Significance level:** alpha = 0.05

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/social_media_ads_clean.csv', parse_dates=['Date'])
print(f'Loaded: {df.shape}')

## H1 - ANOVA: ROI across channels

**H0:** All channel means are equal  
**H1:** At least one channel mean differs

In [ ]:
groups = [grp['ROI'].values for _, grp in df.groupby('Channel_Used')]
f_stat, p_val = stats.f_oneway(*groups)

print('=== ANOVA: ROI across Channels ===')
print(f'F-statistic : {f_stat:.4f}')
print(f'P-value     : {p_val:.6f}')
print()
if p_val < 0.05:
    print('Result: REJECT H0 - ROI differs significantly across channels (p < 0.05).')
else:
    print('Result: FAIL TO REJECT H0')

print('\nChannel mean ROI:')
print(df.groupby('Channel_Used')['ROI'].mean().sort_values(ascending=False).round(4))

## H2 - Pearson Correlation: Acquisition Cost vs ROI

**H0:** No linear relationship (r = 0)  
**H1:** Significant linear relationship exists

In [ ]:
clean = df[['Acquisition_Cost', 'ROI']].dropna()
r, p = stats.pearsonr(clean['Acquisition_Cost'], clean['ROI'])

print('=== Pearson Correlation: Acquisition Cost vs ROI ===')
print(f'r : {r:.4f}')
print(f'p : {p:.6f}')
print()
direction = 'positive' if r > 0 else 'negative'
strength  = 'strong' if abs(r) > 0.5 else 'moderate' if abs(r) > 0.3 else 'weak'
if p < 0.05:
    print(f'Result: REJECT H0 - {strength} {direction} correlation.')
else:
    print('Result: FAIL TO REJECT H0')

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(clean['Acquisition_Cost'], clean['ROI'], alpha=0.3, color='#4361EE', s=15)
m, b = np.polyfit(clean['Acquisition_Cost'], clean['ROI'], 1)
x_line = np.linspace(clean['Acquisition_Cost'].min(), clean['Acquisition_Cost'].max(), 100)
ax.plot(x_line, m * x_line + b, color='#FF6B35', linewidth=2, label=f'r={r:.3f}')
ax.set_xlabel('Acquisition Cost ($)', fontsize=12)
ax.set_ylabel('ROI', fontsize=12)
ax.set_title('Acquisition Cost vs ROI - Pearson Correlation', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/cost_roi_correlation.png', bbox_inches='tight')
plt.show()

## H3 - ANOVA: Conversion Rate across segments

**H0:** All segment means are equal  
**H1:** At least one segment mean differs

In [ ]:
seg_groups = [grp['Conversion_Rate'].values for _, grp in df.groupby('Customer_Segment')]
f2, p2 = stats.f_oneway(*seg_groups)

print('=== ANOVA: Conversion Rate across Segments ===')
print(f'F-statistic : {f2:.4f}')
print(f'P-value     : {p2:.6f}')
print()
if p2 < 0.05:
    print('Result: REJECT H0 - Conversion Rate differs significantly across segments.')
else:
    print('Result: FAIL TO REJECT H0')

print('\nSegment mean Conversion Rate:')
print((df.groupby('Customer_Segment')['Conversion_Rate'].mean() * 100).sort_values(ascending=False).round(2))

## H4 - Pearson Correlation: Engagement Score vs Conversion Rate

In [ ]:
clean2 = df[['Engagement_Score', 'Conversion_Rate']].dropna()
r2, p3 = stats.pearsonr(clean2['Engagement_Score'], clean2['Conversion_Rate'])

print('=== Pearson Correlation: Engagement vs Conversion Rate ===')
print(f'r : {r2:.4f}')
print(f'p : {p3:.6f}')
print()
direction2 = 'positive' if r2 > 0 else 'negative'
strength2  = 'strong' if abs(r2) > 0.5 else 'moderate' if abs(r2) > 0.3 else 'weak'
if p3 < 0.05:
    print(f'Result: REJECT H0 - {strength2} {direction2} correlation.')
else:
    print('Result: FAIL TO REJECT H0')

## Summary table

In [ ]:
summary = pd.DataFrame({
    'Hypothesis': [
        'H1: ROI across channels (ANOVA)',
        'H2: Acq. Cost vs ROI (Pearson)',
        'H3: Conv. Rate across segments (ANOVA)',
        'H4: Engagement vs Conv. Rate (Pearson)'
    ],
    'Statistic': [f'F={f_stat:.3f}', f'r={r:.3f}', f'F={f2:.3f}', f'r={r2:.3f}'],
    'P-value': [p_val, p, p2, p3],
    'Significant (alpha=0.05)': [
        'Yes' if p_val < 0.05 else 'No',
        'Yes' if p     < 0.05 else 'No',
        'Yes' if p2    < 0.05 else 'No',
        'Yes' if p3    < 0.05 else 'No'
    ]
})
summary['P-value'] = summary['P-value'].apply(lambda x: f'{x:.6f}')
display(summary)

---
**End of analysis.** See `/outputs/` for all exported charts.